# 중간보고용 간소 분석 노트북

이 노트북은 중간보고 목적에 맞춰 아래 두 가지를 간단히 보여준다.

1. H.264 IPB/GOP 특성 분석 (핵심)
2. 최소 전송 효율 지표 비교 (보조)

입력 경로
- 비디오: `dataset/videos/`
- 트레이스: `dataset/traces/`
- 산출물: `output/jupyter-notebook/assets/midreport/`


In [ ]:
# cell 1 : 환경 초기화 및 경로 설정
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
for p in [REPO_ROOT, *REPO_ROOT.parents]:
    if (p / '.git').exists():
        REPO_ROOT = p
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tcp_batching_core import PolicyConfig, TransportConfig, VideoTraceConfig, run_simulation

VIDEO_DIR = REPO_ROOT / 'dataset' / 'videos'
TRACE_DIR = REPO_ROOT / 'dataset' / 'traces'
ASSET_DIR = REPO_ROOT / 'output' / 'jupyter-notebook' / 'assets' / 'midreport'

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
TRACE_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
print({'video_dir': str(VIDEO_DIR), 'trace_dir': str(TRACE_DIR), 'asset_dir': str(ASSET_DIR)})


In [ ]:
# cell 2 : 비디오 목록 로드 및 메타정보 구성
video_files = sorted(list(VIDEO_DIR.glob('*.mp4')) + list(VIDEO_DIR.glob('*.mkv')) + list(VIDEO_DIR.glob('*.mov')))

rows = []
for vp in video_files:
    stem = vp.stem
    parts = stem.split('_')
    group = parts[0] if len(parts) >= 1 else 'unknown'
    video_name = parts[1] if len(parts) >= 2 else stem
    resolution = parts[2] if len(parts) >= 3 else 'na'
    bitrate = parts[3] if len(parts) >= 4 else 'na'
    rows.append({
        'video_path': str(vp),
        'group': group,
        'video_name': video_name,
        'resolution': resolution,
        'bitrate': bitrate,
        'motion_level': 'medium',
        'trace_csv': str((TRACE_DIR / f'{stem}_trace.csv')),
    })

video_meta_df = pd.DataFrame(rows)
display(video_meta_df)
print(f'비디오 개수: {len(video_meta_df)}')


In [ ]:
# cell 3 : frame trace 자동 생성(없을 때만)
def ensure_trace(video_path: Path, trace_path: Path, playback_buffer_ms: float = 50.0) -> None:
    if trace_path.exists():
        return
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'extract_video_trace.py'),
        '--input', str(video_path),
        '--output', str(trace_path),
        '--playback-buffer-ms', str(playback_buffer_ms),
    ]
    subprocess.run(cmd, check=True)

for row in video_meta_df.itertuples(index=False):
    ensure_trace(Path(row.video_path), Path(row.trace_csv))

print('trace 생성/확인 완료')


In [ ]:
# cell 4 : IPB/GOP 특성 요약(핵심)
ipb_rows = []
for row in video_meta_df.itertuples(index=False):
    trace_df = pd.read_csv(row.trace_csv)
    frame_count = len(trace_df)
    if frame_count == 0:
        continue
    i_ratio = float((trace_df['frame_type'].str.upper() == 'I').mean())
    p_ratio = float((trace_df['frame_type'].str.upper() == 'P').mean())
    b_ratio = float((trace_df['frame_type'].str.upper() == 'B').mean())
    key_idx = trace_df.index[trace_df['key_frame'] == 1].to_numpy()
    gop_len = float(np.median(np.diff(key_idx))) if len(key_idx) >= 2 else float(frame_count)
    ipb_rows.append({
        'group': row.group,
        'video_name': row.video_name,
        'frame_count': frame_count,
        'i_ratio': i_ratio,
        'p_ratio': p_ratio,
        'b_ratio': b_ratio,
        'median_gop_len_frames': gop_len,
        'mean_payload_bytes': float(trace_df['payload_bytes'].mean()),
    })

ipb_summary_df = pd.DataFrame(ipb_rows).sort_values(['group', 'video_name'])
display(ipb_summary_df)
ipb_summary_df.to_csv(ASSET_DIR / 'midreport_ipb_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = ipb_summary_df.melt(id_vars=['group', 'video_name'], value_vars=['i_ratio', 'p_ratio', 'b_ratio'], var_name='frame_class', value_name='ratio')
sns.barplot(data=plot_df, x='video_name', y='ratio', hue='frame_class', ax=ax)
ax.set_title('비디오별 IPB 비율')
ax.set_xlabel('video_name')
ax.set_ylabel('ratio')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(ASSET_DIR / 'midreport_ipb_ratio.png')
plt.show()


In [ ]:
# cell 5 : 최소 전송효율 지표 비교(보조)
transport = TransportConfig(rtt_ms=10, bandwidth_mbps=5, delayed_ack_ms=40)
policies = [
    PolicyConfig('heuristic_frame_aware'),
    PolicyConfig('frame_action_adaptive', available_paths=2),
]

eff_rows = []
for row in video_meta_df.itertuples(index=False):
    workload = VideoTraceConfig(trace_csv_path=Path(row.trace_csv), playback_buffer_ms=50.0, loop_count=1)
    for policy in policies:
        result = run_simulation(workload, transport, policy)
        eff_rows.append({
            'group': row.group,
            'video_name': row.video_name,
            'policy': policy.name,
            'late_frame_ratio': result['late_frame_ratio'],
            'dropped_frame_ratio': result['dropped_frame_ratio'],
            'useful_goodput_bytes': result['useful_goodput_bytes'],
            'decodable_gop_ratio': result['decodable_gop_ratio'],
        })

eff_summary_df = pd.DataFrame(eff_rows).sort_values(['group', 'video_name', 'policy'])
display(eff_summary_df)
eff_summary_df.to_csv(ASSET_DIR / 'midreport_transport_efficiency.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=eff_summary_df, x='video_name', y='late_frame_ratio', hue='policy', ax=ax)
ax.set_title('정책별 Late Frame 비율')
ax.set_xlabel('video_name')
ax.set_ylabel('late_frame_ratio')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(ASSET_DIR / 'midreport_late_frame_ratio.png')
plt.show()


## 중간보고 작성 체크리스트

- `midreport_ipb_summary.csv`에서 그룹별 IPB/GOP 특성 차이 정리
- `midreport_transport_efficiency.csv`에서 정책별 late/drop/goodput 비교 정리
- 해석은 "IPB 특성이 전송 행동에 주는 영향" 중심으로 3~5줄 요약
